# Random Forest for Heart Failure Prediction
> **Dataset:** Heart Failure Clinical Records (UCI, 299 patients, 12 features)  
> **Target:** `DEATH_EVENT` (binary: 0 = survived, 1 = died)  
> **Author:** Pranav Ananth  
> **Reference:** Chicco & Jurman (2020), BMC Medical Informatics

---
## Algorithm Overview
**Random Forest** is a bagging ensemble of decision trees. Each tree is trained on a bootstrap sample, and at each node only a random subset of features is considered for splitting:

$$\hat{y} = \text{majority\_vote}\left(T_1(x), T_2(x), \ldots, T_B(x)\right)$$

This double randomization (bootstrap + feature subsampling) decorrelates the trees, drastically reducing variance. Random Forest consistently delivers strong performance on tabular medical data and provides reliable feature importances via mean decrease in impurity.

## 1 · Setup & Imports

In [ ]:
# Install required packages
!pip install -q pandas numpy scikit-learn matplotlib seaborn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    matthews_corrcoef, roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix, roc_curve,
    precision_recall_curve, classification_report, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ All packages imported successfully")

## 2 · Load & Explore Data

In [ ]:
# ── Option A: upload file manually ──────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_excel(list(uploaded.keys())[0])

# ── Option B: load directly from UCI ────────────────────────────────────────
import urllib.request
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00519/heart_failure_clinical_records_dataset.csv"
urllib.request.urlretrieve(url, "heart_failure.csv")
df = pd.read_csv("heart_failure.csv")

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Quick EDA
print("=== Class Distribution ===")
print(df['DEATH_EVENT'].value_counts())
print(f"\nClass imbalance ratio: {df['DEATH_EVENT'].value_counts()[0]/df['DEATH_EVENT'].value_counts()[1]:.2f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df['DEATH_EVENT'].value_counts().plot(kind='bar', ax=axes[0], color=['#3266ad','#e05a4a'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_xticklabels(['Survived (0)', 'Died (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Feature correlation heatmap
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=axes[1], cmap='RdBu_r', center=0,
            linewidths=0.5, annot=False, fmt='.2f')
axes[1].set_title('Feature Correlation Matrix', fontsize=13)

plt.tight_layout()
plt.show()

## 3 · Train / Test Split

In [ ]:
# Feature / target split
feature_cols = [c for c in df.columns if c != 'DEATH_EVENT']
X = df[feature_cols]
y = df['DEATH_EVENT']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")

## 4 · Evaluation Utilities

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "MCC":       round(matthews_corrcoef(y_test, y_pred), 4),
        "AUC-ROC":   round(roc_auc_score(y_test, y_proba), 4),
        "F1":        round(f1_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
    }

    print(f"\n{'='*50}")
    print(f"  {model_name} — Test Set Results")
    print(f"{'='*50}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v}")
    print()
    print(classification_report(y_test, y_pred, target_names=['Survived','Died']))
    return metrics, y_pred, y_proba

def plot_results(model, X_test, y_test, y_pred, y_proba, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'{model_name} — Evaluation Plots', fontsize=14, fontweight='bold')

    # Confusion matrix
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                           display_labels=['Survived','Died']).plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Confusion Matrix')

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[1].plot(fpr, tpr, color='#3266ad', lw=2, label=f'AUC = {auc:.3f}')
    axes[1].plot([0,1],[0,1],'--', color='gray', lw=1)
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve'); axes[1].legend()

    # Precision-Recall curve
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    axes[2].plot(rec, prec, color='#e05a4a', lw=2)
    axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
    axes[2].set_title('Precision-Recall Curve')
    axes[2].axhline(y=y_test.mean(), linestyle='--', color='gray', lw=1, label='Baseline')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

def plot_learning_curve(model, X, y, model_name):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        train_sizes=np.linspace(0.1, 1.0, 8), scoring='roc_auc', n_jobs=-1
    )
    plt.figure(figsize=(8, 4))
    plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#3266ad', label='Train AUC')
    plt.fill_between(train_sizes,
                     train_scores.mean(1)-train_scores.std(1),
                     train_scores.mean(1)+train_scores.std(1), alpha=0.1, color='#3266ad')
    plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#e05a4a', label='CV Val AUC')
    plt.fill_between(train_sizes,
                     val_scores.mean(1)-val_scores.std(1),
                     val_scores.mean(1)+val_scores.std(1), alpha=0.1, color='#e05a4a')
    plt.xlabel('Training Set Size'); plt.ylabel('AUC-ROC')
    plt.title(f'{model_name} — Learning Curve'); plt.legend()
    plt.tight_layout(); plt.show()

print("✅ Evaluation utilities ready")

In [ ]:
def cross_validate_model(model, X, y, model_name):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scorers = ['matthews_corrcoef','roc_auc','f1','precision','recall','accuracy']
    print(f"\n5-Fold Cross-Validation — {model_name}")
    print("-" * 40)
    for s in scorers:
        scores = cross_val_score(model, X, y, cv=cv, scoring=s)
        print(f"  {s:<22}: {scores.mean():.4f} ± {scores.std():.4f}")

print("✅ Cross-validation utility ready")

## 5 · Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,       # fully grown trees — variance controlled by bagging
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',  # sqrt(n_features) per split
    class_weight='balanced',  # handles 32% positive class
    random_state=42,
    n_jobs=-1
)

metrics, y_pred, y_proba = evaluate_model(
    rf_model, X_train, X_test, y_train, y_test, "Random Forest"
)

## 6 · Evaluation Plots

In [ ]:
plot_results(rf_model, X_test, y_test, y_pred, y_proba, "Random Forest")

## 7 · Cross-Validation

In [ ]:
cross_validate_model(
    RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    X, y, "Random Forest"
)

## 8 · Feature Importance (MDI + Permutation)

In [ ]:
from sklearn.inspection import permutation_importance

rf_model.fit(X_train, y_train)

# MDI importance
mdi = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()

# Permutation importance
perm = permutation_importance(rf_model, X_test, y_test, n_repeats=30,
                               random_state=42, scoring='roc_auc')
perm_s = pd.Series(perm.importances_mean, index=feature_cols).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
mdi.plot(kind='barh', ax=axes[0], color=['#3266ad' if v>0.05 else '#a0b8d8' for v in mdi])
axes[0].set_title('MDI (Mean Decrease Impurity)', fontsize=12)
perm_s.plot(kind='barh', ax=axes[1], color=['#4a9e6b' if v>0.01 else '#a0c8b0' for v in perm_s])
axes[1].set_title('Permutation Importance (AUC-ROC)', fontsize=12)
plt.tight_layout(); plt.show()

## 9 · n_estimators Sweep

In [ ]:
n_est_range = [10, 25, 50, 100, 200, 300, 500]
val_aucs = []
for n in n_est_range:
    s = cross_val_score(
        RandomForestClassifier(n_estimators=n, class_weight='balanced',
                               random_state=42, n_jobs=-1),
        X, y, cv=5, scoring='roc_auc'
    ).mean()
    val_aucs.append(s)
    print(f"  n={n:<5}: CV AUC = {s:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(n_est_range, val_aucs, 'o-', color='#4a9e6b', lw=2)
plt.xlabel('Number of trees'); plt.ylabel('CV AUC-ROC')
plt.title('Random Forest — Number of Trees vs. Performance')
plt.tight_layout(); plt.show()

## 10 · Out-of-Bag Error

In [ ]:
rf_oob = RandomForestClassifier(n_estimators=300, oob_score=True,
                                   class_weight='balanced', random_state=42, n_jobs=-1)
rf_oob.fit(X_train, y_train)
print(f"OOB accuracy estimate: {rf_oob.oob_score_:.4f}")
oob_proba = rf_oob.oob_decision_function_[:, 1]
print(f"OOB AUC-ROC estimate:  {roc_auc_score(y_train, oob_proba):.4f}")

## 11 · Learning Curve

In [ ]:
plot_learning_curve(
    RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    X, y, "Random Forest"
)